# Guardrails

*Notebook 20*

Guardrails add enforceable checks around an agent run.

These non-streamed examples reject unwanted input and withhold flagged final responses.

Misplaced checks can allow costly work or a response the application should block.

---

## 🔧 Setup

**Cost:** The main path invokes `Runner.run()` eleven times.

That total is eight top-level calls plus three nested TopicChecker calls.

Under the expected verdicts, eight model calls must finish.

A ninth parallel main request starts, and its response is discarded.

It may finish or be canceled, and may consume tokens.

This notebook does not measure provider billing.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from pydantic import BaseModel

# Notebook-specific imports
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrail,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
    output_guardrail,
)

MODEL = "gpt-5-mini"


print("✅ Ready!")

---

## 🎯 The Problem

Agents may answer requests outside their intended scope.

Instructions guide behavior.

Guardrails reject input or discard output when a tripwire fires.

Rule-based checks run in Python.

LLM-based checks use another model and can make mistakes.

When a tripwire fires, the SDK raises instead of returning the response.

---

## 🛡️ Part 1: How Guardrails Work

Agent-level guardrails check the run at two boundaries:
```
Default (parallel) input mode:
User Input ─┬─> [Input Guardrail]   ← both start together
            └─> Agent runs

Blocking mode (Part 5):
User Input → [Input Guardrail] → Agent runs

Agent output → [Output Guardrail] → Response delivered
```

**Parallel mode**

When a **tripwire** fires, the agent's result is discarded.

The agent may already have spent tokens or run tools.

A fast local guardrail can win the race before a provider call begins.

Only blocking mode guarantees that order.

**Multi-agent runs**

Input guardrails apply to the starting agent.

Output guardrails apply to the final agent.

**Function tool calls**

Function tools can use `tool_input_guardrail` and `tool_output_guardrail`.

This separate API is not covered in this course.

---

## ✅ Part 2: Rule-Based Input Guardrail

A rule-based guardrail checks input in Python.

It needs no LLM.

Use it for clear conditions: keywords, length limits, formats, or allow/block lists.

Use an LLM-based guardrail (Part 3) when the check needs judgment.

In [ ]:
# Every input guardrail receives the run context, the agent, and the input.
# Ignore any parameters your check doesn't need.

# A simple keyword-based guardrail
@input_guardrail
async def topic_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    """Block requests containing selected off-topic keywords."""
    input_text = input if isinstance(input, str) else str(input)

    # Off-topic keywords
    off_topic_terms = ["politics", "stocks", "weather", "sports", "news"]
    is_off_topic = any(term in input_text.lower() for term in off_topic_terms)

    return GuardrailFunctionOutput(
        tripwire_triggered=is_off_topic,
        # output_info travels with the guardrail result and the tripwire
        # exception: log it explicitly when you need it recorded.
        # A string, or a Pydantic model (Part 4) for structured detail.
        output_info="Off-topic request detected" if is_off_topic else "OK"
    )


cooking_instructions = (
    "You are a cooking assistant. "
    "Help with recipes, techniques, and ingredients."
)

# The @input_guardrail decorator turns the function into a guardrail.
# To set run_in_parallel, pass it to the decorator
# (@input_guardrail(run_in_parallel=False), the form Lesson 24 uses)
# or wrap the function in InputGuardrail(...), shown in Part 5.
cooking_agent = Agent(
    name="CookingAssistant",
    instructions=cooking_instructions,
    model=MODEL,
    input_guardrails=[topic_guardrail]
)

# --------------------------------------------------------------
print("✅ Cooking agent with input guardrail ready")

#### Test: On-Topic Request

In [ ]:
print("🧪 RULE-BASED GUARDRAIL TEST")
print("=" * 60)

# On-topic: passes
try:

    result = await Runner.run(
        cooking_agent,
        input="How do I make a good pasta sauce?"
    )

    guardrail_results = result.input_guardrail_results
    on_topic_ok = (
        len(guardrail_results) == 1
        and guardrail_results[0].output.output_info == "OK"
    )
    print(f"{'✅' if on_topic_ok else '❌'} Guardrail evidence: "
          f"{[item.output.output_info for item in guardrail_results]}")
    if not on_topic_ok:
        raise RuntimeError("On-topic guardrail evidence was missing or unexpected.")

    print("✅ On-topic request passed")
    print(f"Response: {result.final_output[:160]}")
except InputGuardrailTripwireTriggered:
    print("❌ Unexpectedly blocked")
    raise RuntimeError(
        "On-topic request was blocked: this keyword rule is deterministic."
    )

#### Test: Off-Topic Request

In [ ]:
# Off-topic: blocked
try:

    result = await Runner.run(
        cooking_agent,
        input="What are today's top news stories?"
    )

    print("Response:", result.final_output)
    raise RuntimeError(
        "Off-topic request was not blocked: this keyword rule is deterministic."
    )
except InputGuardrailTripwireTriggered as e:
    reason = e.guardrail_result.output.output_info
    print("🛑 Input blocked by guardrail")
    print(f"   Reason: {reason}")
    if reason != "Off-topic request detected":
        raise RuntimeError(f"Unexpected guardrail output_info: {reason}")

print("=" * 60)

---

## 🤖 Part 3: LLM-Based Input Guardrail

For checks that require judgment, a guardrail can run a separate agent.

This demo uses `MODEL` with a structured `TopicCheck` verdict.

In [ ]:
class TopicCheck(BaseModel):
    is_cooking_related: bool
    reasoning: str


topic_checker_instructions = (
    "Determine if the user's message is related to cooking, recipes, food, "
    "or ingredients.\n"
    "Return is_cooking_related=True only if it's clearly about cooking or food."
)

topic_checker_agent = Agent(
    name="TopicChecker",
    instructions=topic_checker_instructions,
    model=MODEL,
    output_type=TopicCheck
)


@input_guardrail
async def llm_topic_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    """Use an LLM to check if input is cooking-related."""

    # Reuse the same application context inside the checker agent.
    # The nested run joins the active trace on its own.
    result = await Runner.run(topic_checker_agent, input, context=ctx.context)

    check = result.final_output
    return GuardrailFunctionOutput(
        tripwire_triggered=not check.is_cooking_related,
        output_info=check.reasoning
    )


smart_cooking_agent = Agent(
    name="SmartCookingAssistant",
    instructions=cooking_instructions,
    model=MODEL,
    input_guardrails=[llm_topic_guardrail]
)

# --------------------------------------------------------------
print("✅ Smart cooking agent with LLM guardrail ready")

#### Test: Ambiguous Request

In [ ]:
print("🤖 LLM GUARDRAIL TEST")
print("=" * 60)

test_cases = [
    ("What herbs pair well with chicken?", False),   # expected: pass
    ("Can you write me a Python script?", True),      # expected: block
    ("What wine goes with pasta?", False),            # expected: pass (food-adjacent)
]

for test_input, expected_blocked in test_cases:
    try:

        result = await Runner.run(smart_cooking_agent, input=test_input)

        blocked = False
        reasoning = result.input_guardrail_results[0].output.output_info
        preview = result.final_output[:100]
    except InputGuardrailTripwireTriggered as e:
        blocked = True
        reasoning = e.guardrail_result.output.output_info
        preview = None

    # ⚠️ not raise on mismatch: the classifier verdict is model-generated
    agree = blocked == expected_blocked
    icon = "✅" if agree else "⚠️"
    print(f"{icon} {'BLOCKED' if blocked else 'PASSED'} "
          f"(expected {'block' if expected_blocked else 'pass'}): {test_input}")
    print(f"   Classifier reasoning: {reasoning}")
    if preview:
        print(f"   → {preview}")
    print()

print("=" * 60)

### 💡 Key Takeaway

LLM guardrails handle cases fixed rules miss, but their verdicts remain model-generated.

---

## 📤 Part 4: Output Guardrail

Output guardrails inspect the complete final response.

In these non-streamed runs, a tripwire prevents `Runner.run()` from returning it.

They can withhold responses flagged by sensitive-data, policy, or quality checks.

Streaming applications may forward raw events before final output validation.

Output guardrails cannot undo token use or tool side effects.

This demo uses a rule-based word count.

Exercise 2 uses an LLM-based tone check.

In [ ]:
class LengthCheck(BaseModel):
    is_acceptable: bool
    word_count: int


@output_guardrail
async def length_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str
) -> GuardrailFunctionOutput:
    """Block responses over 30 words: enforce concise answers."""
    word_count = len(output.split())
    is_too_long = word_count > 30

    return GuardrailFunctionOutput(
        tripwire_triggered=is_too_long,
        output_info=LengthCheck(
            is_acceptable=not is_too_long,
            word_count=word_count
        )
    )


concise_agent = Agent(
    name="ConciseAssistant",
    instructions="You are a helpful assistant. Answer questions clearly.",
    model=MODEL,
    output_guardrails=[length_guardrail]
)

# --------------------------------------------------------------
print("✅ Concise agent with output guardrail ready (30 word limit)")

#### Test: Short vs Long Response

In [ ]:
print("📤 OUTPUT GUARDRAIL TEST")
print("=" * 60)

# Deterministic rule probe first: prove both branches without a model call.
# Pass real callback arguments: guardrails receive a RunContextWrapper and Agent.
probe_ctx = RunContextWrapper(context=None)
short_check = await length_guardrail.guardrail_function(
    probe_ctx,
    concise_agent,
    "Four words is fine"
)
long_check = await length_guardrail.guardrail_function(
    probe_ctx,
    concise_agent,
    " ".join(["word"] * 31)
)
print(
    "Rule probe: "
    f"4 words trips={short_check.tripwire_triggered}, "
    f"31 words trips={long_check.tripwire_triggered}"
)
if short_check.tripwire_triggered or not long_check.tripwire_triggered:
    raise RuntimeError("Length rule misbehaved on fixed inputs.")

print()

# Short answer: should pass
try:

    result = await Runner.run(concise_agent, input="What is 2 + 2?")

    observed_words = len(result.final_output.split())
    output_results = result.output_guardrail_results
    short_evidence_ok = (
        len(output_results) == 1
        and isinstance(output_results[0].output.output_info, LengthCheck)
        and output_results[0].output.output_info.is_acceptable
        and output_results[0].output.output_info.word_count == observed_words
    )
    print(f"{'✅' if short_evidence_ok else '❌'} Output guardrail evidence: "
          f"{[item.output.output_info for item in output_results]}")
    if not short_evidence_ok:
        raise RuntimeError(
            "Short-response guardrail evidence was missing or inconsistent."
        )

    print(f"✅ Short response passed ({observed_words} words)")
    print(f"   {result.final_output}")
except OutputGuardrailTripwireTriggered as e:
    check = e.guardrail_result.output.output_info
    print(
        "⚠️ Inconclusive: the model produced "
        f"{check.word_count} words, and the guardrail correctly blocked it"
    )

print()

# Long-response attempt: the model may or may not actually write 80 words,
# so distinguish "answered briefly" from "over-limit response escaped".
try:

    result2 = await Runner.run(
        concise_agent,
        input="Write at least 80 words explaining the history of pasta."
    )

    word_count = len(result2.final_output.split())
    # Final backstop: no over-limit value should return successfully after
    # the earlier output-guardrail evidence passed.
    if word_count > 30:
        raise RuntimeError(
            f"Over-limit response escaped the guardrail ({word_count} words)."
        )
    print(
        "⚠️ Inconclusive: the model answered briefly "
        f"({word_count} words), nothing to block"
    )
except OutputGuardrailTripwireTriggered as e:
    check = e.guardrail_result.output.output_info
    print(
        "✅ Output guardrail fired: "
        f"{check.word_count} words exceeded the 30-word limit"
    )

print("=" * 60)

---

## ⚙️ Part 5: Blocking Mode

Blocking mode changes when an input guardrail runs.

With `run_in_parallel=False`, it finishes before the main agent starts.

Use it when the check is cheap and avoiding the main call matters.

The fast Python rule in Part 2 may also win its scheduling race.

Only blocking mode makes pre-model rejection a guarantee.

Blocked requests in blocking mode spend no main-agent tokens.

Parallel mode lets passing requests overlap work.

In [ ]:
# Same topic guardrail, but run_in_parallel=False runs it BEFORE the agent.
# An off-topic request is rejected before the agent (or its tokens) ever start.
# Equivalent shorthand: decorate with @input_guardrail(run_in_parallel=False).
blocking_topic_guardrail = InputGuardrail(
    topic_guardrail.guardrail_function,
    run_in_parallel=False
)

blocking_agent = Agent(
    name="CookingAssistant_Blocking",
    instructions=cooking_instructions,
    model=MODEL,
    input_guardrails=[blocking_topic_guardrail],
)

# --------------------------------------------------------------
print("✅ Blocking-mode agent ready")

#### Test: Blocking Mode

In [ ]:
print("⚙️ BLOCKING MODE TEST")
print("=" * 60)

attached_guardrails = blocking_agent.input_guardrails
blocking_config_ok = (
    len(attached_guardrails) == 1
    and attached_guardrails[0] is blocking_topic_guardrail
    and attached_guardrails[0].run_in_parallel is False
)
print(f"{'✅' if blocking_config_ok else '❌'} Blocking configuration attached")
if not blocking_config_ok:
    raise RuntimeError("Blocking guardrail configuration is missing or inconsistent.")

# Off-topic request: guardrail runs first, agent never starts.
try:

    result = await Runner.run(
        blocking_agent,
        input="What's the latest news in sports?"
    )

    print(f"Response: {result.final_output}")
    raise RuntimeError(
        "Off-topic request was not blocked in blocking mode: "
        "this rule is deterministic."
    )
except InputGuardrailTripwireTriggered as e:
    tripwire_matches = e.guardrail_result.guardrail is attached_guardrails[0]
    blocking_ok = blocking_config_ok and tripwire_matches
    print(f"{'✅' if blocking_ok else '❌'} Demo check passed")
    if not blocking_ok:
        raise RuntimeError("The tripwire did not come from the attached guardrail.")
    print("🛑 Blocked: guardrail ran first, agent never started")
    print("   No tokens spent on main agent")

print("=" * 60)

⚠️ **Session note:** A tripwire does not erase submitted input.

With `session=`, the SDK can persist the input before the guardrail raises.

Redact sensitive data before `Runner.run()` (Lesson 18).

---

## 💪 Practice Exercises

### Exercise 1: ASCII-Only Input Guardrail

*Covers: input guardrails (rule-based heuristics)*

Build an input guardrail that blocks messages containing non-ASCII characters.

This demonstrates a rule-based heuristic, not reliable language detection.

The exercise makes three run attempts.

Two expected pass cases reach the model.

The blocked default-parallel case can race cancellation.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: ASCII-Only Input Guardrail
# --------------------------------------------------------------
# Objective: Block messages containing non-ASCII characters.

# TODO 1: Create an @input_guardrail function that:
#           - Checks if the input contains only ASCII characters
#           - Triggers the tripwire if non-ASCII characters are found
#           (This is a simple heuristic, not perfect, but teaches the pattern)

# TODO 2: Create an Agent with this guardrail attached

# TODO 3: Test with:
#           - "Hello, how are you?" (should pass)
#           - "Bonjour, comment allez-vous?" (passes: pure ASCII)
#           - "こんにちは" (should block, Japanese uses non-ASCII)
#           - Print whether each passed or was blocked, and raise on any
#             unexpected result: this rule is deterministic

# --- Write your code below this line ---

### Exercise 2: Sentiment Output Guardrail

*Covers: output guardrails (tone filtering)*

Build an output guardrail that blocks responses with a negative or harsh tone.

Completing the exercise makes four model calls.

Two are test-agent runs, and two are nested tone-checker runs.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Sentiment Output Guardrail
# --------------------------------------------------------------
# Objective: Block agent responses that are harsh or dismissive.

# TODO 1: Create a Pydantic model ToneCheck with:
#           - is_appropriate: bool
#           - reasoning: str

# TODO 2: Create a tone_checker_agent that classifies
#           whether a response is appropriate and professional
#           output_type=ToneCheck

# TODO 3: Create an @output_guardrail that runs the tone checker
#           and triggers the tripwire if is_appropriate=False

# TODO 4: Create a test agent with this output guardrail and EXACT behaviors:
#           instructions: if the message contains PASS, reply exactly
#           "I'd be happy to help." If it contains BLOCK, reply exactly
#           "That's a stupid question. Figure it out yourself."

# TODO 5: Run one PASS message and one BLOCK message:
#           compare expected vs observed for each, print the classifier's
#           reasoning, and mark mismatches with ⚠️ rather than raising
#           (the LLM classifier's verdict is model-generated)

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Guardrails run on input or output:**

- `input_guardrails` validate the starting agent's input.

- `output_guardrails` check the final agent's complete response.

- In non-streamed runs, an output tripwire withholds the final result.

- Function-tool guardrails are a separate SDK API.
<br>
<br>

**Implement guardrails with rules or an LLM:**

- Rule-based checks are fast and fit clear conditions.

- LLM-based checks handle nuance, but can be wrong and cost another model call.
<br>
<br>

**Tripwires halt execution:**

- `tripwire_triggered=True` raises the matching tripwire exception.

- Handle expected tripwires explicitly around `Runner.run()`.
<br>
<br>

**Blocking mode guarantees pre-model rejection:**

- **Default parallel mode:** The guardrail and agent overlap.

- A tripped main request may finish or be canceled. Its response is discarded.

- **Blocking mode:** The guardrail finishes before the main agent starts.

- Use blocking when the check is cheap and avoiding the main call matters.

---

## 📍 Next Step

**Notebook 21: Prompt Injection & Tool Safety**  

Learn how attackers exploit agents through injected instructions.

Then build defenses into your tool design.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-20-guardrails)

---